# Quilt1M vs Quilt1M_curated Bimodal Bridge Comparison

Create comparative barplots showing AUROC retrieval performance for bimodal bridge models trained with quilt1m vs quilt1m_curated datasets, combined with secondary datasets (hest1k and cellxgene_census__archs4_geo)

In [ ]:
# Parameters from Snakemake
datasets = snakemake.params.datasets
model_mappings_original = snakemake.params.model_mappings_original
model_mappings_curated = snakemake.params.model_mappings_curated
modality_colors = snakemake.params.modality_colors
comparison_results_dir = snakemake.params.comparison_results_dir

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from pathlib import Path

# Apply matplotlib style
plt.style.use(snakemake.input.mpl_style)

PAIRS = ["transcriptome_text", "image_text", "image_transcriptome", "right_left"]

print(f"Test datasets: {datasets}")
print(f"Original model mappings: {model_mappings_original}")
print(f"Curated model mappings: {model_mappings_curated}")

In [ ]:
# Load AUROC data for each bimodal bridge model (exactly like original)
original_auroc_data = []
curated_auroc_data = []
dataset_labels = []

# Process original models (with quilt1m)
for i, dataset in enumerate(datasets):
    # Load retrieval results
    df = pd.read_csv(snakemake.input.original_retrieval_results[i])

    # Extract AUROC metrics (average of left-right and right-left)
    for pair in PAIRS:
        try:
            left_right_auroc = df[f"test_retrieval/{pair}/rocauc_macroAvg"].iloc[-1]
        except KeyError:
            continue
        else:
            flipped = "_".join(pair.split("_")[::-1])
            right_left_auroc = df[f"test_retrieval/{flipped}/rocauc_macroAvg"].iloc[-1]
            break
    else:
        raise ValueError(f"none of the pairs found for {dataset}")

    # Average the two directions
    avg_auroc = (left_right_auroc + right_left_auroc) / 2
    original_auroc_data.append(avg_auroc)
    
    # Create dataset labels (same as original)
    dataset_labels.append(pair.replace("_", "-") + "\n(" +
        dataset.replace("_", " ").replace("__", " + ").title() + ")"
    )

# Process curated models (with quilt1m_curated)
for i, dataset in enumerate(datasets):
    # Load retrieval results
    df = pd.read_csv(snakemake.input.curated_retrieval_results[i])

    # Extract AUROC metrics (average of left-right and right-left)
    for pair in PAIRS:
        try:
            left_right_auroc = df[f"test_retrieval/{pair}/rocauc_macroAvg"].iloc[-1]
        except KeyError:
            continue
        else:
            flipped = "_".join(pair.split("_")[::-1])
            right_left_auroc = df[f"test_retrieval/{flipped}/rocauc_macroAvg"].iloc[-1]
            break
    else:
        raise ValueError(f"none of the pairs found for {dataset}")

    # Average the two directions
    avg_auroc = (left_right_auroc + right_left_auroc) / 2
    curated_auroc_data.append(avg_auroc)

print(f"Original AUROC values: {original_auroc_data}")
print(f"Curated AUROC values: {curated_auroc_data}")
print(f"Dataset labels: {dataset_labels}")

In [ ]:
# Prepare data for grouped bar plot (horizontal like original)
n_datasets = len(datasets)
y_pos = np.arange(n_datasets)
bar_height = 0.35

print(f"Original values: {original_auroc_data}")
print(f"Curated values: {curated_auroc_data}")

In [ ]:
# Create the grouped horizontal barplot (like original)
fig, ax = plt.subplots(figsize=(3, 1.7))  # Same size as original

# Define colors for each dataset (matching the modality pairings)
# Use the same color logic as the original bimodal_bridge_plot
colors = [
    modality_colors["text-transcriptome"],
    modality_colors["image-transcriptome"],
    modality_colors["text-image"],
]

# Create horizontal grouped bars
bars1 = ax.barh(
    y_pos - bar_height/2,
    original_auroc_data,
    bar_height,
    color=colors,
    alpha=0.8,
    edgecolor="black",
    linewidth=1,
    label="Quilt1M"
)

bars2 = ax.barh(
    y_pos + bar_height/2,
    curated_auroc_data,
    bar_height,
    color=colors,
    alpha=0.6,
    edgecolor="black",
    linewidth=1,
    hatch='///',  # Pattern to distinguish curated
    label="Quilt1M_curated"
)

# Add dotted line for random baseline (0.5)
ax.axvline(
    x=0.5, color="gray", linestyle="--", linewidth=2, alpha=0.7, label="Random Baseline"
)

# Add value labels on bars
for bar, value in zip(bars1, original_auroc_data):
    ax.text(
        bar.get_width() + 0.01,
        bar.get_y() + bar.get_height() / 2,
        f"{value:.3f}",
        ha="left",
        va="center",
        fontsize=8
    )

for bar, value in zip(bars2, curated_auroc_data):
    ax.text(
        bar.get_width() + 0.01,
        bar.get_y() + bar.get_height() / 2,
        f"{value:.3f}",
        ha="left",
        va="center",
        fontsize=8
    )

# Customize plot (same as original)
ax.set_yticks(y_pos)
ax.set_yticklabels(dataset_labels)
ax.set_xlabel("AUROC")
ax.set_title(
    "Bimodal Bridge Model AUROC Performance\nby Evaluation Dataset",
)

# Set x-axis limits to show the full range
ax.set_xlim(0, 1.0)

# Add grid for better readability
ax.grid(axis="x", alpha=0.3)
ax.set_axisbelow(True)

# Add legend
ax.legend(loc="upper right")

# Adjust layout and save both PNG and SVG
plt.tight_layout()
plt.savefig(snakemake.output.plot, dpi=300, bbox_inches="tight")
plt.savefig(snakemake.output.plot_svg, bbox_inches="tight")
plt.show()

In [ ]:
# Calculate and print differences
print("\nPerformance Differences (Curated - Original):")
for i, dataset in enumerate(datasets):
    diff = curated_auroc_data[i] - original_auroc_data[i]
    pct_change = (diff / original_auroc_data[i]) * 100
    print(f"{dataset}: {diff:+.4f} ({pct_change:+.2f}%)")

print(f"\nOriginal models used:")
for dataset in datasets:
    print(f"  {dataset} -> {model_mappings_original[dataset]['bimodal_bridge']}")

print(f"\nCurated models used:")
for dataset in datasets:
    print(f"  {dataset} -> {model_mappings_curated[dataset]['bimodal_bridge']}")